# Equations and root finding

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain what a root-finding problem is and formulate equations as $f(x)=0$
2. explain the theoretical basis of the bisection method and Newton's method
3. implement the methods using simple Python code
4. use a tolerance to control a numerical calculation
5. discuss strengths, weaknesses and convergence of the methods
6. use ready-made root solvers in SciPy for chemical problems
```

## Equations as root-finding problems

Solving an equation and finding a root are really the same problem written in two different ways. If we have

$$g(x)=h(x),$$

we can move everything to one side:

$$f(x)=g(x)-h(x)=0.$$

A **root** is a value of $x$ for which the function value is zero. Solving $g(x)=h(x)$ therefore means finding the value of $x$ that makes $f(x)=0$. This is what we mean when we say that we **formulate the equation as a root-finding problem**.

For quadratic equations we know a dedicated formula. For more complicated equations there is not always a practical analytical expression for the solution. Numerical methods are more general: instead, they approach the solution step by step.


## A chemical example: pH of a weak acid

We use a 0.010 M solution of acetic acid as an example. For a monoprotic weak acid, we can combine the mass balance for the protonated and deprotonated forms, $C=[\mathrm{HA}]+[\mathrm{A^-}]$, with the acid dissociation constant to write

$$[\mathrm{A^-}]=C\frac{K_a}{[\mathrm{H_3O^+}]+K_a}.$$

```{admonition} Where does the expression for $[\mathrm{A^-}]$ come from?
:class: tip, dropdown

$C$ is the total analytical concentration. The acid does not disappear; it is distributed between two forms:

$$C=[\mathrm{HA}]+[\mathrm{A^-}].$$

This is the mass balance. Note that $C\neq[\mathrm{HA}]$ unless almost none of the acid has dissociated.

The acid dissociation constant gives $[\mathrm{HA}]=\dfrac{[\mathrm{H_3O^+}][\mathrm{A^-}]}{K_a}$. Substituting this into the mass balance and factoring out $[\mathrm{A^-}]$ gives

$$C=[\mathrm{A^-}]\left(\frac{[\mathrm{H_3O^+}]}{K_a}+1\right)=[\mathrm{A^-}]\,\frac{[\mathrm{H_3O^+}]+K_a}{K_a},$$

which gives the expression above when solved for $[\mathrm{A^-}]$. The fraction is the proportion of the acid present in the deprotonated form at a given pH. At $\mathrm{pH}=\mathrm{p}K_a$, it is $1/2$.
```

The charge balance is

$$[\mathrm{H_3O^+}]=[\mathrm{A^-}]+[\mathrm{OH^-}],$$

and the ion product of water gives

$$[\mathrm{OH^-}]=\frac{K_w}{[\mathrm{H_3O^+}]}.$$

If we set $h=[\mathrm{H_3O^+}]$, the whole problem can be collected in one function:

$$f(h)=h-C\frac{K_a}{h+K_a}-\frac{K_w}{h}.$$

The pH is found when the charge balance is satisfied, that is, when $f(h)=0$. In other words, we do not have to isolate $h$ algebraically. It is enough to evaluate $f(h)$ and search for its root.

Before solving an equation numerically, it is often useful to **visualise the function**. A graph can show us approximately where the root is, whether several roots exist, and which starting values might be sensible:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 0.010
Ka = 1.75e-5
Kw = 1.0e-14

def charge_balance(h):
    A_minus = C * Ka / (h + Ka)
    OH = Kw / h
    return h - A_minus - OH

h = np.logspace(-7, -2, 500)
plt.semilogx(h, charge_balance(h))
plt.axhline(0)
plt.xlabel(r"$[\mathrm{H_3O^+}]$ (mol/L)")
plt.ylabel("Charge balance")
plt.show()

## From graph to algorithm

To locate the root more precisely, we need an algorithm. A very simple idea is to start at a value $x$ and move along the graph with a fixed step size $dx$. At each step, we compare the signs of $f(x)$ and $f(x+dx)$. If the sign changes, a root must lie between the points as long as the function is continuous.

```{image} images/root_scan.png
:width: 500px
:align: center
```

In the figure, $f(x_7)$ and $f(x_8)$ have opposite signs. The root must therefore lie somewhere between $x_7$ and $x_8$, and we can use the midpoint as a first estimate.

This approach is intuitive, but not very efficient. If $dx$ is large, the answer is coarse. If $dx$ is small, we must examine many points. The idea of a **sign change** is nevertheless important because it leads directly to a more robust and efficient method: the *bisection method*.


In [ ]:
def f(x):
    return x**2 - x - 2

x = -5
x_end = 5
dx = 0.5

while x < x_end and f(x)*f(x + dx) > 0:
    x = x + dx

root = (x + (x + dx))/2
print("A first estimate is x =", root)


## The bisection method

Instead of moving through the whole interval with equally sized steps, we can be smarter. We start with an interval $[a,b]$ where $f(a)$ and $f(b)$ have opposite signs. We then divide the interval in two and **discard the half that cannot contain the root**.

We begin with the simplest possible code. Here we deliberately choose a function with a root that the algorithm can hit exactly, so that the idea is easy to follow.


In [ ]:
def f(x):
    return 2*x - 2

a = -5
b = 5
m = (a + b)/2

while f(m) != 0:
    if f(a)*f(m) < 0:
        b = m
    elif f(b)*f(m) < 0:
        a = m
    m = (a + b)/2

print("The root is x =", m)


Study the code line by line. Each iteration makes the interval half as wide. This is why the method is called the **bisection method**.

More generally, the method works as follows:

1. Choose an interval $[a,b]$ where $f(a)$ and $f(b)$ have opposite signs.
2. Find the midpoint

$$m=\frac{a+b}{2}.$$

3. Determine which half, $[a,m]$ or $[m,b]$, still contains a sign change.
4. Keep that half and repeat.

```{image} images/bisection.png
:width: 500px
:align: center
```

The figure shows two iterations. The point is not that we know the root beforehand, but that we always know **which half it must lie in**.

### From exact equality to a tolerance

In real numerical problems, we should not wait for `f(m) == 0`. Floating-point arithmetic and complicated functions mean that we may never hit zero exactly. Instead, we decide how close to zero is good enough. This is called a **tolerance**.


In [ ]:
def f(x):
    return x**2 - x - 2

a = 0
b = 5
tolerance = 1E-8
m = (a + b)/2

while abs(f(m)) > tolerance:
    if f(a)*f(m) < 0:
        b = m
    elif f(b)*f(m) < 0:
        a = m
    m = (a + b)/2

print("The root is x =", m)
print("f(x) =", f(m))


Now that we understand the algorithm as a concrete loop, it is natural to wrap it in a function so that we can use it on many problems without rewriting the code.


In [ ]:
def bisection(f, a, b, tol=1E-10, max_iterations=100):
    i = 0
    m = (a + b)/2

    while i < max_iterations and abs(f(m)) > tol:
        if f(a)*f(m) < 0:
            b = m
        elif f(b)*f(m) < 0:
            a = m
        m = (a + b)/2
        i = i + 1

    if i == max_iterations:
        print("The maximum number of iterations has been reached.")

    return m, i


We can now use the same function for our pH problem:


In [ ]:
h_root, iterations = bisection(charge_balance, 1e-7, 1e-2)
pH = -np.log10(h_root)

print(f"[H3O+] = {h_root:.6e} mol/L")
print(f"pH = {pH:.3f}")
print("Iterations:", iterations)


### Try it yourself

Complete the bisection method in the editor and use it to find the pH of the weak acid.

<iframe src="../../basthon/?from=examples/equations_bisection.py" width="100%" height="600" frameborder="0" title="Try it yourself: the bisection method" loading="lazy" allowfullscreen></iframe>

## Newton's method

The bisection method makes use of the fact that a root lies **between** two points. Newton's method uses a different idea: the tangent at one point can be used to predict where the root lies.

1. Choose a starting guess $x_0$.
2. Draw, or imagine, the tangent at $(x_0,f(x_0))$.
3. Find where the tangent crosses the x-axis. This becomes the next guess, $x_1$.
4. Repeat the process from the new point.

```{image} images/newton_method.png
:width: 500px
:align: center
```

The figure shows why the method can approach a root much faster than bisection.

Let us derive the formula. The tangent through $(x_n,f(x_n))$ has slope $f'(x_n)$:

$$y=f(x_n)+f'(x_n)(x-x_n).$$

We want the root of the tangent, so we set $y=0$:

$$0=f(x_n)+f'(x_n)(x-x_n).$$

Solving for $x$ gives the next estimate:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Again, we start with the simplest possible implementation before making a general function.


In [ ]:
def f(x):
    return x**2 - x - 2

def f_derivative(x):
    return 2*x - 1

x = 5
tolerance = 1E-8

while abs(f(x)) > tolerance:
    x = x - f(x)/f_derivative(x)

print("The root is x =", x)


Once the algorithm is clear as a simple loop, we can wrap it in a function:


In [ ]:
def newton_method(f, f_derivative, x, tol=1E-10):
    while abs(f(x)) > tol:
        x = x - f(x)/f_derivative(x)
    return x


Newton's method needs only one starting guess and often converges quickly. The disadvantage is that we need the derivative, and an unfortunate starting guess can lead us to the wrong root or prevent the method from converging. At this stage, it is more important to **understand the limitation** than to build extensive error handling into the first implementation.

Later, when we use ready-made library functions, we get more robustness and information about whether the method actually converged.

```{admonition} Exercise along the way
:class: tip
Try several different starting values for a function with more than one root. Do all starting guesses lead to the same root? What does this tell you about the difference between bisection and Newton's method?
```

## Ready-made solvers in SciPy

Once we understand the principle, it is common to use tested algorithms from numerical libraries. `scipy.optimize.root_scalar` provides several methods for one-dimensional root-finding problems.


In [ ]:
from scipy.optimize import root_scalar

bisect_result = root_scalar(charge_balance, bracket=[1e-7, 1e-2], method="bisect")

def d_charge_balance(h):
    return 1 + C*Ka/(h + Ka)**2 + Kw/h**2

newton_result = root_scalar(
    charge_balance, x0=4e-4, fprime=d_charge_balance, method="newton"
)

print("Bisection:")
print("  converged:", bisect_result.converged)
print("  iterations:", bisect_result.iterations)
print("  pH:", -np.log10(bisect_result.root))

print("\nNewton:")
print("  converged:", newton_result.converged)
print("  iterations:", newton_result.iterations)
print("  pH:", -np.log10(newton_result.root))

## Which method should we choose?

| Situation | A natural choice |
|---|---|
| We know an interval containing a sign change | Bisection or another bracketed method |
| We have a good starting value and know the derivative | Newton |
| We want a robust ready-made solver | `root_scalar` with an appropriate method |
| There may be several roots | Plot or scan the interval first |

The important point is not only to obtain a number, but to check that the number actually solves the chemical problem.

```{admonition} Numerical workflow
:class: important
1. Formulate the chemistry as $f(x)=0$.
2. Inspect the function and choose a sensible search interval.
3. Choose a method, tolerance and, if needed, a starting guess.
4. Check that the method converged.
5. Substitute the solution back into the model and assess whether it is chemically reasonable.
```

## Short summary

- Equations can be formulated as root-finding problems.
- The bisection method is robust when we have a sign change.
- Newton's method can be fast, but is more sensitive to the starting guess and derivative.
- A tolerance and a maximum number of iterations make the calculation controllable.
- SciPy provides ready-made solvers, but we should still understand the problem we give them.

## Exercises

```{admonition} Exercise 1 – pH of a weak acid
:class: tip
Use `bisection` to find the pH of 0.0250 M acetic acid with $K_a=1.75\cdot10^{-5}$. Compare with the approximation $[\mathrm{H_3O^+}]\approx\sqrt{K_aC}$. How large is the difference?
```

```{admonition} Exercise 2 – choose a method
:class: tip
You need to solve three problems:

1. A function has a known sign change between 2 and 3, but its derivative is difficult to calculate.
2. You know a good starting value and both $f(x)$ and $f'(x)$ are easy to calculate.
3. You suspect that the function has three roots in the interval $[-5,5]$.

Choose an approach for each case and justify your choices.
```

```{admonition} Exercise 3 – several roots
:class: tip
Find all solutions of $x^5=5x^3+3$. First plot the root function. Then use the bisection method or `root_scalar` on suitable subintervals.
```

```{admonition} Exercise 4 – Newton and the starting guess
:class: tip
Investigate $f(x)=x^3-2x+2$ with Newton's method. Test at least five different starting values. Explain why the same method can succeed from one starting value and fail from another.
```

```{admonition} Exercise 5 – chemical equilibrium
:class: tip
For the reaction $\mathrm{A \rightleftharpoons B}$, we start with 1.00 M A and 0 M B. At equilibrium, $[B]=x$ and $[A]=1-x$. Let $K=3.5$ and formulate $K=[B]/[A]$ as a root-finding problem. Find $x$ numerically and check the solution analytically.
```

```{admonition} Exercise 6 – temperature at which a process changes spontaneity
:class: tip
Assume that $\Delta H=45.0$ kJ/mol and $\Delta S=125$ J/(mol K) are constant over a temperature interval. Formulate $\Delta G(T)=\Delta H-T\Delta S=0$ as a root-finding problem and find the temperature. This equation is easy to solve analytically, so use that result to check the numerical method.
```